# 02 · 云遮挡退化扫描（阶段 B · H4）

**目标**：在 M1 模型上做云遮挡率扫描（0% → 70%），产出"遮挡率 vs 解译精度"响应曲线，验证 H4（存在可定位的失效拐点）。

**前置**：
1. 本 notebook 基于 M1（01_terratorch_eurosat_baseline.ipynb）已训练出的模型。
2. 需要把 eo-degrade 仓库的 `degrade/` 目录作为 Dataset 添加到本 notebook（右侧 + Add Input，选 **Dataset 类型**）。
3. EuroSAT 数据（M1 已下载到 data/ 或通过 Add Input 添加）。

In [ ]:
# 环境准备
!pip install -q terratorch torchgeo 2>&1 | tail -1
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 挂载退化库（从 Kaggle Input 或仓库目录）
import sys, os, glob

# 尝试从 Kaggle Input 找 degrade 目录
candidates = []
for root in ['/kaggle/input', '.']:
    for hit in glob.glob(os.path.join(root, '**', 'degrade', '__init__.py'), recursive=True):
        candidates.append(os.path.dirname(hit))

if candidates:
    d = candidates[0]
    sys.path.insert(0, os.path.dirname(d))
    print('找到 degrade 库:', d)
else:
    raise FileNotFoundError('未找到 degrade 库。请通过右侧 + Add Input 添加 eo-degrade 仓库（Dataset 类型）。')

from degrade.cloud import add_cloud_field, add_cloud_mask
print('云退化库加载成功')

In [ ]:
# 加载 M1 模型（两种模式二选一）
from terratorch.tasks import ClassificationTask

# ---- 模式 1：从 M1 checkpoint 加载（推荐，精度 89.4%）----
CKPT_PATH = None  # 改为你的 checkpoint 路径
# 没有 checkpoint 也可用 glob 自动找
if CKPT_PATH is None:
    hits = glob.glob('/kaggle/**/*.ckpt', recursive=True)
    if hits:
        CKPT_PATH = hits[0]
        print('自动找到 checkpoint:', CKPT_PATH)

MODEL_ARGS = dict(
    model_args={
        'backbone': 'prithvi_eo_v1_100',
        'backbone_kwargs': {'pretrained': True, 'num_frames': 1, 'bands': ['BLUE','GREEN','RED']},
        'decoder': 'IdentityDecoder',
        'num_classes': 10,
    },
    model_factory='EncoderDecoderFactory',
    loss='ce',
    lr=1e-4,
)

if CKPT_PATH is not None:
    try:
        model = ClassificationTask.load_from_checkpoint(CKPT_PATH, map_location='cpu')
        print('从 checkpoint 加载成功（精度应为 M1 水平 ~89%）')
    except Exception as e:
        print('checkpoint 加载失败，回退 pretrained 模式:', e)
        model = ClassificationTask(**MODEL_ARGS)
else:
    # ---- 模式 2：pretrained 快速验证（无 checkpoint 时跑通 pipeline）----
    print('未找到 checkpoint，使用 pretrained 模型（精度较低，仅验证流程）')
    model = ClassificationTask(**MODEL_ARGS)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print('模型就绪:', device)

In [ ]:
# 加载 EuroSAT 测试集（与 M1 相同的 datamodule 配置）
from terratorch.datamodules import EuroSATDataModule

dm = EuroSATDataModule(
    root='data',
    batch_size=32,
    num_workers=2,
    bands=['BLUE', 'GREEN', 'RED'],
    test_transform=[{'class_path': 'albumentations.Resize', 'init_args': {'height': 224, 'width': 224}}],
)
dm.setup('test')
loader = dm.test_dataloader()
print('测试集 batch 数:', len(loader))

# 值域诊断（TerraTorch 输出可能是 0-1 或 0-255，退化函数要求 [0,1]）
xb = next(iter(loader))['image']
print('image 范围: min=%.3f max=%.3f dtype=%s' % (xb.min().item(), xb.max().item(), xb.dtype))
GLOBAL_MAX = xb.max().item()

In [ ]:
# 推理辅助函数（兼容 TerraTorch 输出形态）
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad()
def eval_cloud(model, loader, frac, seed=0):
    """在遮挡率 frac 下评估精度/F1，返回 (acc, f1, actual_mask_frac)"""
    all_pred, all_label, mask_fracs = [], [], []
    for batch in loader:
        img = batch['image'].to(device)
        y = batch['label'].to(device)
        # 值域自适应：>1.5 视为 0-255，先归一化再退化
        scale = 1.0
        if GLOBAL_MAX > 1.5:
            img = img / 255.0
            scale = 255.0
        deg, mask = add_cloud_mask(img, cloud_fraction=frac, seed=seed)
        if scale != 1.0:
            deg = deg * scale
        out = model(deg)
        if isinstance(out, dict):
            out = out.get('logits', list(out.values())[0])
        all_pred.append(out.argmax(dim=1).cpu())
        all_label.append(y.cpu())
        mask_fracs.append(mask.mean().item())
    pred = torch.cat(all_pred).numpy()
    lab = torch.cat(all_label).numpy()
    return accuracy_score(lab, pred), f1_score(lab, pred, average='macro'), float(np.mean(mask_fracs))

In [ ]:
# 云遮挡率扫描（0% → 70%）
FRACTIONS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
SEED = 0
results = []

print('遮挡率 | 精度 | F1 | 实际mask比例')
print('-' * 50)
for frac in FRACTIONS:
    acc, f1, mfrac = eval_cloud(model, loader, frac, seed=SEED)
    results.append({'cloud_fraction': frac, 'accuracy': acc, 'f1': f1, 'actual_mask_frac': mfrac})
    print(f'{frac:.0%}   | {acc:.4f} | {f1:.4f} | {mfrac:.3f}')

# 保存 CSV（Kaggle 输出目录）
import csv
os.makedirs('/kaggle/working/results', exist_ok=True)
csv_path = '/kaggle/working/results/cloud_scan.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['cloud_fraction', 'accuracy', 'f1', 'actual_mask_frac'])
    w.writeheader()
    w.writerows(results)
print('\nCSV 已保存:', csv_path)

In [ ]:
# H4 拐点分析（加速下降拐点 + 失效点）
fractions = [r['cloud_fraction'] for r in results]
accs = [r['accuracy'] for r in results]

def find_breakpoint(fractions, accs, min_drop=0.02):
    for i in range(1, len(accs)):
        drop = accs[i - 1] - accs[i]
        if drop >= min_drop:
            return fractions[i], drop
    return None, None

def find_failure_point(fractions, accs, keep_ratio=0.7):
    baseline = accs[0]
    for frac, acc in zip(fractions, accs):
        if acc < baseline * keep_ratio:
            return frac, acc
    return None, None

bf, drop = find_breakpoint(fractions, accs)
ff, fa = find_failure_point(fractions, accs)

print(f'干净精度: {accs[0]:.3f}')
if bf is not None:
    print(f'[H4] 加速下降拐点: 遮挡率 {bf:.0%}（该步下降 {drop:.3f}）')
else:
    print('[H4] 曲线平滑下降，未检测到加速拐点（阴性结果同样有价值）')
if ff is not None:
    print(f'[H4] 失效点: 遮挡率 {ff:.0%}（精度 {fa:.3f} < 干净×70%）')
else:
    print('[H4] 全程未跌破干净精度的 70%（模型对云遮挡鲁棒）')

In [ ]:
# 响应曲线可视化 + 保存 PNG
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([f * 100 for f in fractions], [a * 100 for a in accs], 'o-', color='#2B5FB8', linewidth=2, label='Accuracy')
if bf is not None:
    ax.axvline(bf * 100, color='#EA6668', linestyle='--', linewidth=1.5, label=f'拐点 {bf:.0%}')
if ff is not None:
    ax.axvline(ff * 100, color='#FAAD14', linestyle='--', linewidth=1.5, label=f'失效点 {ff:.0%}')
ax.axhline(accs[0] * 100 * 0.7, color='#6B7280', linestyle=':', linewidth=1, label='70% 基线')
ax.set_xlabel('云遮挡率 (%)')
ax.set_ylabel('解译精度 (%)')
ax.set_title('云遮挡率 vs 解译精度（Prithvi-EO-1.0 · EuroSAT）')
ax.legend()
ax.grid(alpha=0.3)
png_path = '/kaggle/working/results/cloud_scan_curve.png'
plt.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print('PNG 已保存:', png_path)

## 结果回填

跑完后把 `cloud_scan.csv` 和曲线图下载下来，提交到仓库：
- `results/cloud_scan.csv`（实验记录）
- `results/cloud_scan_curve.png`（README 用图）
- 在 README 更新 H4 结论（拐点/失效点数值，或阴性结果）

**可复现性**：`SEED=0` 固定；云退化 `add_cloud_mask(img, cloud_fraction=frac, seed=0)`。